In [ ]:
import whisper

model = whisper.load_model("base")

# Note the 'r' right before the string
video_path = r"D:\KRSL_OnlineSchool\KRSL_OnlineSchool_final_dataset\video\elarna_video_sorted_2020_2021\grade_01\subject_01\01_02_1_01_02.mp4"

result = model.transcribe(video_path)
print("Transcript: ", result["text"])

Transcript:   Убрай день дорогие ребята! Я рада наши встречи! Тема нашего урока, кто такой, экскуссобот. Сегодня вы ответите на вопросы по тексту. Вы выполните слога звуковой аналий слов со звуком «Э». Напишите слоги и слова сострочной и заглавной буквами «Э». Ребята, скажите, вы любите путешествовать. Во время путешествия мы часто ходим на экскурсии. А знаете ли вы, что это такое? Экскурсия – это коллективное посещение определенных объектов спознавательной или научной целью. Сегодня на уроке мы отправимся на экскурсию. Но отправимся не просто так. Что-то посмотреть о зановыми знаниями. Давайте откроем страницы 106 наших учебников и отгадаем загадки. Он весь город нам покажет. Да еще о нем расскажет. По музеям проведет. Человек верно. Экскурсовод, а кто это? Давайте прочитаем определение этого слова. Экскурсовод – специалист по проведению экскурсий. Он должен хорошо знать историю, культуру и географию того места, где проводит экскурсию. А какое первый звук мы слышим? Словие экскурсовод

In [ ]:
print(result["segments"])

[{'id': 0, 'seek': 0, 'start': 0.0, 'end': 4.0, 'text': ' Убрай день дорогие ребята! Я рада наши встречи!', 'tokens': [50364, 6523, 1552, 481, 3183, 13509, 24365, 5299, 37678, 0, 4857, 26622, 386, 36314, 25669, 435, 0, 50564], 'temperature': 0.0, 'avg_logprob': -0.3688893610117387, 'compression_ratio': 1.805668016194332, 'no_speech_prob': 0.08750216662883759}, {'id': 1, 'seek': 0, 'start': 4.0, 'end': 8.0, 'text': ' Тема нашего урока, кто такой, экскуссобот.', 'tokens': [50564, 44064, 386, 45309, 1595, 481, 20296, 11, 12278, 13452, 11, 13817, 4218, 7071, 461, 2061, 1631, 13, 50764], 'temperature': 0.0, 'avg_logprob': -0.3688893610117387, 'compression_ratio': 1.805668016194332, 'no_speech_prob': 0.08750216662883759}, {'id': 2, 'seek': 0, 'start': 8.0, 'end': 14.0, 'text': ' Сегодня вы ответите на вопросы по тексту.', 'tokens': [50764, 35913, 2840, 25284, 5878, 1470, 48418, 2801, 1069, 34879, 585, 13, 51064], 'temperature': 0.0, 'avg_logprob': -0.3688893610117387, 'compression_ratio': 1.

In [12]:
import os
import pandas as pd
import whisper
from pathlib import Path

def transcribe_dataset(video_folder_path, output_dir):
    # 1. Setup paths and directories
    video_root = Path(video_folder_path)
    output_root = Path(output_dir)
    segments_dir = output_root / "transcripts_detailed"
    
    segments_dir.mkdir(parents=True, exist_ok=True)
    master_csv_path = output_root / "master_transcripts.csv"
    
    # 2. Load Whisper Model (change to 'medium' or 'large' if needed)
    print("Loading Whisper model...")
    model = whisper.load_model("base", device="cuda")
    
    # 3. Load existing progress if available (to allow resuming)
    processed_files = set()
    master_data = []
    
    if master_csv_path.exists():
        print("Found existing master CSV. Loading progress to skip completed videos...")
        try:
            df_existing = pd.read_csv(master_csv_path)
            processed_files = set(df_existing['video_path'].tolist())
            master_data = df_existing.to_dict('records')
        except Exception as e:
            print(f"Could not read existing CSV, starting fresh. Error: {e}")

    # 4. Find all video files recursively (.mp4, .mkv, .avi)
    video_extensions = ["*.mp4", "*.mkv", "*.avi", "*.mov"]
    video_files = []
    for ext in video_extensions:
        video_files.extend(list(video_root.rglob(ext)))
        
    print(f"Found {len(video_files)} total videos. {len(processed_files)} already processed.")

    # 5. Loop through each video
    for i, video_path in enumerate(video_files, 1):
        # Convert to string path for compatibility
        video_path_str = str(video_path)
        filename = video_path.name
        filename_stem = video_path.stem  # filename without extension
        
        if video_path_str in processed_files:
            continue
            
        print(f"[{i}/{len(video_files)}] Transcribing: {filename}")
        
        try:
            # Run transcription
            result = model.transcribe(video_path_str)
            full_text = result["text"].strip()
            
            # --- Save Individual Segment/Timestamp CSV ---
            segment_records = []
            for segment in result["segments"]:
                segment_records.append({
                    "start_time": segment["start"],
                    "end_time": segment["end"],
                    "phrase": segment["text"].strip()
                })
            
            df_segments = pd.DataFrame(segment_records)
            segment_file_path = segments_dir / f"{filename_stem}_segments.csv"
            df_segments.to_csv(segment_file_path, index=False, encoding='utf-8-sig')
            
            # --- Append to Master Data List ---
            master_data.append({
                "filename": filename,
                "video_path": video_path_str,
                "full_transcript": full_text
            })
            
            # Progressively update Master CSV so you don't lose data if it crashes
            df_master = pd.DataFrame(master_data)
            df_master.to_csv(master_csv_path, index=False, encoding='utf-8-sig')
            
        except Exception as e:
            print(f"❌ Error processing {filename}: {e}")
            continue

    print(f"\n🎉 Done! All transcriptions saved to {output_root}")

# --- EXECUTION ---
if __name__ == "__main__":
    # Replace these paths with your real folder paths
    INPUT_FOLDER = r"D:\KRSL_OnlineSchool\KRSL_OnlineSchool_final_dataset\video"
    OUTPUT_FOLDER = r"D:\KRSL_OnlineSchool\Transcription_Outputs"

In [13]:
transcribe_dataset(INPUT_FOLDER, OUTPUT_FOLDER)

Loading Whisper model...
Found existing master CSV. Loading progress to skip completed videos...
Found 4563 total videos. 18 already processed.
[19/4563] Transcribing: 04_06_06_16_05.mp4
[20/4563] Transcribing: 04_07_1_03_05.mp4
[21/4563] Transcribing: 04_07_2_06_05.mp4
[22/4563] Transcribing: 04_07_4_07_05.mp4
[23/4563] Transcribing: 04_09_1_03_05.mp4
[24/4563] Transcribing: 04_12_1_03_05.mp4
[25/4563] Transcribing: 04_13_6_17_05.mp4
❌ Error processing 04_13_6_17_05.mp4: [Errno 22] Invalid argument: 'D:\\KRSL_OnlineSchool\\Transcription_Outputs\\transcripts_detailed\\04_13_6_17_05_segments.csv'
[26/4563] Transcribing: 04_14_3_3_05.mp4
❌ Error processing 04_14_3_3_05.mp4: Failed to load audio: ffmpeg version 6.1.1 Copyright (c) 2000-2023 the FFmpeg developers
  built with clang version 20.1.8
  configuration: --prefix=/c/miniconda3/conda-bld/ffmpeg_1762437060025/_h_env/Library --cc=clang.exe --ar=llvm-ar --nm=llvm-nm --ranlib=llvm-ranlib --strip= --disable-doc --enable-swresample --ena

KeyboardInterrupt: 